# Advanced: Multi-Strategy Utility Metrics with PAMOLA.CORE

**Goal**: Master all 3 utility evaluation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Basic models (logistic, linear) with core metrics
- **Strategy 2**: Advanced models (RF, SVM/SVR) with comprehensive metrics
- **Strategy 3**: Ensemble comparison with cross-validation
- Analyze privacy-utility tradeoffs
- Compare model performance across strategies
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of ML model evaluation
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── metrics/
        └── 02_utility_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.metrics.operations.utility_ops import UtilityMetricOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load original and transformed datasets (1000 records each)
- Auto-creates sample if files not found
- Review both data structures before proceeding

**What you'll see:**
- Load status for both datasets (from file or generated)
- Dataset overviews (1000 records, 7 columns)
- Sample records (first 5 rows) from each
- Column statistics (types, ranges, unique counts)
- Transformation info (noise added for privacy)

**Dataset features:**
- 1000 records with realistic distributions
- Classification target (3 classes: 0, 1, 2)
- Regression target (continuous values)
- Transformed data includes privacy noise

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
original_data_path = examples_dir / 'data_examples' / 'utility_complex_original.csv'
transformed_data_path = examples_dir / 'data_examples' / 'utility_complex_transformed.csv'

print(f"📂 Looking for data at: {original_data_path.parent}\n")

# Load or generate original data
if original_data_path.exists():
    print(f"📂 Loading original data from: {original_data_path}")
    original_df = read_csv(original_data_path)
    print(f"✓ Loaded {len(original_df)} records from file")
else:
    print("📊 Generating synthetic original dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic data with correlations
    age = np.random.randint(18, 80, n_records)
    education_years = np.random.randint(8, 20, n_records)
    hours_worked = np.random.randint(20, 60, n_records)
    
    # Income correlated with education and hours
    income = (
        30000 + 
        education_years * 2000 + 
        hours_worked * 500 + 
        np.random.normal(0, 5000, n_records)
    ).astype(int)
    
    # Classification target (income brackets)
    classification_target = pd.cut(
        income, 
        bins=[0, 40000, 60000, np.inf], 
        labels=[0, 1, 2]
    ).astype(int)
    
    # Regression target (years of experience)
    regression_target = (
        (age - 18) * 0.6 + 
        education_years * 0.5 + 
        np.random.normal(0, 3, n_records)
    )
    
    original_df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'age': age,
        'income': income,
        'education_years': education_years,
        'hours_worked': hours_worked,
        'classification_target': classification_target,
        'regression_target': regression_target
    })
    
    # Save for future use
    try:
        original_data_path.parent.mkdir(parents=True, exist_ok=True)
        original_df.to_csv(original_data_path, index=False)
        print(f"✓ Generated and saved to: {original_data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {original_data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

# Load or generate transformed data
if transformed_data_path.exists():
    print(f"\n📂 Loading transformed data from: {transformed_data_path}")
    transformed_df = read_csv(transformed_data_path)
    print(f"✓ Loaded {len(transformed_df)} records from file")
else:
    print("\n📊 Generating transformed dataset with privacy noise...\n")
    
    np.random.seed(43)
    
    # Create transformed data by adding noise
    transformed_df = original_df.copy()
    
    # Add privacy noise to numeric fields
    transformed_df['age'] = transformed_df['age'] + np.random.randint(-3, 4, len(transformed_df))
    transformed_df['income'] = transformed_df['income'] + np.random.normal(0, 2000, len(transformed_df)).astype(int)
    transformed_df['education_years'] = transformed_df['education_years'] + np.random.randint(-1, 2, len(transformed_df))
    transformed_df['hours_worked'] = transformed_df['hours_worked'] + np.random.randint(-3, 4, len(transformed_df))
    transformed_df['regression_target'] = transformed_df['regression_target'] + np.random.normal(0, 2, len(transformed_df))
    
    # Ensure targets remain unchanged (these are ground truth)
    transformed_df['classification_target'] = original_df['classification_target']
    
    # Save for future use
    try:
        transformed_data_path.parent.mkdir(parents=True, exist_ok=True)
        transformed_df.to_csv(transformed_data_path, index=False)
        print(f"✓ Generated and saved to: {transformed_data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {transformed_data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Original Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(original_df):,}")
print(f"  Columns: {', '.join(original_df.columns)}")

print(f"\n🔍 Original Sample Records:")
print(original_df.head())

print(f"\n📊 Transformed Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(transformed_df):,}")
print(f"  Columns: {', '.join(transformed_df.columns)}")

print(f"\n🔍 Transformed Sample Records:")
print(transformed_df.head())

print(f"\n📈 Column Statistics (Original):")
for col in original_df.columns:
    if col == 'record_id':
        continue
    dtype_str = str(original_df[col].dtype)
    if pd.api.types.is_numeric_dtype(original_df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): min={original_df[col].min():.2f}, max={original_df[col].max():.2f}, mean={original_df[col].mean():.2f}")

print(f"\n💡 Perfect datasets for testing all 3 utility evaluation strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit target field names for each metric type
   - `classification_target`: For classification metrics (default: "classification_target")
   - `regression_target`: For regression metrics (default: "regression_target")
2. **Review feature columns**: Verify columns list excludes ID and target fields
3. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each target field)
- Feature columns list (age, income, education_years, hours_worked)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource with both datasets ready (✓)

**Note:** All fields must exist in both datasets before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "classification_target": "classification_target",   # For classification metrics
    "regression_target": "regression_target"             # For regression metrics
}

# Feature columns (exclude ID and target fields)
FEATURE_COLUMNS = ['age', 'income', 'education_years', 'hours_worked', FIELD_CONFIG["classification_target"], FIELD_CONFIG["regression_target"]]

# Validate all fields exist in both datasets
print("📋 Validating field configuration...\n")
for metric_type, field_name in FIELD_CONFIG.items():
    if field_name not in original_df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {metric_type} not found in original dataset!\n"
            f"Available columns: {', '.join(original_df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    if field_name not in transformed_df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {metric_type} not found in transformed dataset!\n"
            f"Available columns: {', '.join(transformed_df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {metric_type:25s}: '{field_name}'")

# Validate feature columns
for col in FEATURE_COLUMNS:
    if col not in original_df.columns or col not in transformed_df.columns:
        raise ValueError(f"❌ Feature column '{col}' not found in datasets!")

print(f"\n✓ Feature columns: {', '.join(FEATURE_COLUMNS)}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="utility_advanced_001",
    task_type="multi_strategy_utility",
    description="Multi-strategy utility metrics evaluation",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource with both datasets
data_source = DataSource(
    dataframes={
        "original_dataset": original_df,
        "transformed_dataset": transformed_df
    }
)
print("✓ DataSource created with both datasets")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Basic Models - Core Metrics

**How to use:**
- Uses simple models (logistic, linear) for fast evaluation
- Focuses on core metrics only
- Run to execute basic model strategy

**Key parameters:**
- **Classification**: Logistic regression with accuracy, F1
- **Regression**: Linear regression with R², MAE
- `cv_folds=1`: Single train-test split (fast)
- `normalize=True`: Feature normalization enabled

**What you'll see:**
- Configuration summary
- Progress bar: validation → training → evaluation → metrics → viz → save
- Completion time (2-5 seconds expected)
- Artifact count (4-6 files)

**Validate:**
- ✅ Execution time <10 seconds
- ✅ Both classification and regression completed
- ✅ Metrics generated for both types
- ✅ Status = "completed"

**Note:** Best for quick baseline evaluation before trying advanced models

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: BASIC MODELS - CORE METRICS")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Basic Models",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = UtilityMetricOperation(
    name='utility_basic',
    
    # Metric types
    utility_metrics=['classification', 'regression'],
    
    # Metric-specific parameters
    metric_params={
        'classification': {
            'models': ['logistic'],
            'metrics': ['accuracy', 'f1'],
            'cv_folds': 1,
            'test_size': 0.2,
            'value_field': FIELD_CONFIG['classification_target']
        },
        'regression': {
            'models': ['linear'],
            'metrics': ['r2', 'mae'],
            'cv_folds': 1,
            'test_size': 0.2,
            'value_field': FIELD_CONFIG['regression_target']
        }
    },
    
    # Feature columns
    columns=FEATURE_COLUMNS,
    
    # Processing settings
    normalize=True,
    confidence_level=0.95,
    use_dask=False,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Classification: Logistic (accuracy, f1)")
print(f"  Regression: Linear (r2, mae)")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_basic',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Store metrics for comparison
if result_s1.metrics:
    metrics_s1 = result_s1.metrics
    if 'classification' in metrics_s1:
        print(f"\n📊 Classification (Logistic):")
        for metric, value in metrics_s1['classification'].get('logistic', {}).items():
            if isinstance(value, (int, float)):
                print(f"   {metric}: {value:.4f}")
    
    if 'regression' in metrics_s1:
        print(f"\n📊 Regression (Linear):")
        for metric, value in metrics_s1['regression'].get('linear', {}).items():
            if isinstance(value, (int, float)):
                print(f"   {metric}: {value:.4f}")

## STRATEGY 2: Advanced Models - Comprehensive Metrics

**How to use:**
- Uses advanced models (RF, SVM/SVR) for robust evaluation
- Includes comprehensive metric set
- Run to execute advanced model strategy

**Key parameters:**
- **Classification**: Random Forest + SVM with accuracy, F1, ROC-AUC, PR curves
- **Regression**: Random Forest + SVR with R², MAE, MSE, RMSE
- `cv_folds=1`: Single train-test split for speed
- `test_size=0.2`: 80-20 split

**What you'll see:**
- Configuration summary
- Progress bar: validation → training → evaluation → metrics → viz → save
- Completion time (5-15 seconds expected)
- Artifact count (6-10 files)

**Validate:**
- ✅ Execution time <30 seconds
- ✅ Multiple models evaluated
- ✅ Comprehensive metrics computed
- ✅ Status = "completed"

**Note:** More robust than basic models but slower. Best for production deployment

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: ADVANCED MODELS - COMPREHENSIVE METRICS")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Advanced",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = UtilityMetricOperation(
    name='utility_advanced',
    utility_metrics=['classification', 'regression'],
    metric_params={
        'classification': {
            'models': ['rf', 'svm'],
            'metrics': ['accuracy', 'f1', 'roc_auc', 'precision_recall_tradeoff'],
            'cv_folds': 1,
            'test_size': 0.2,
            'value_field': FIELD_CONFIG['classification_target']
        },
        'regression': {
            'models': ['rf', 'svr'],
            'metrics': ['r2', 'mae', 'mse', 'rmse'],
            'cv_folds': 1,
            'test_size': 0.2,
            'value_field': FIELD_CONFIG['regression_target']
        }
    },
    columns=FEATURE_COLUMNS,
    normalize=True,
    use_dask=False,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Classification: RF + SVM (accuracy, f1, roc_auc, PR curves)")
print(f"  Regression: RF + SVR (r2, mae, mse, rmse)")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_advanced',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Store metrics
if result_s2.metrics:
    metrics_s2 = result_s2.metrics
    if 'classification' in metrics_s2:
        print(f"\n📊 Classification Results:")
        for model in ['rf', 'svm']:
            if model in metrics_s2['classification']:
                print(f"\n   {model.upper()}:")
                for metric, value in metrics_s2['classification'][model].items():
                    if isinstance(value, (int, float)):
                        print(f"     {metric}: {value:.4f}")
    
    if 'regression' in metrics_s2:
        print(f"\n📊 Regression Results:")
        for model in ['rf', 'svr']:
            if model in metrics_s2['regression']:
                print(f"\n   {model.upper()}:")
                for metric, value in metrics_s2['regression'][model].items():
                    if isinstance(value, (int, float)):
                        print(f"     {metric}: {value:.4f}")

## STRATEGY 3: Ensemble Analysis - Cross-Validation

**How to use:**
- Combines all models with robust cross-validation
- Uses 5-fold CV for reliable performance estimates
- Run to execute ensemble analysis strategy

**Key parameters:**
- **All models**: Logistic, RF, SVM for classification; Linear, RF, SVR for regression
- **All metrics**: Complete metric sets for both types
- `cv_folds=5`: 5-fold cross-validation (robust)
- `stratified=True`: Stratified sampling for classification

**What you'll see:**
- Configuration summary
- Progress bar: validation → CV training → evaluation → metrics → viz → save
- Completion time (10-30 seconds expected)
- Artifact count (10-15 files)

**Validate:**
- ✅ Execution time <60 seconds
- ✅ All models evaluated
- ✅ CV-averaged metrics computed
- ✅ Status = "completed"

**Note:** Most reliable but slowest. Recommended for final evaluation and reporting

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: ENSEMBLE ANALYSIS - CROSS-VALIDATION")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Ensemble",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = UtilityMetricOperation(
    name='utility_ensemble',
    utility_metrics=['classification', 'regression'],
    metric_params={
        'classification': {
            'models': ['logistic', 'rf', 'svm'],
            'metrics': ['accuracy', 'f1', 'roc_auc', 'precision_recall_tradeoff'],
            'cv_folds': 5,
            'stratified': True,
            'value_field': FIELD_CONFIG['classification_target']
        },
        'regression': {
            'models': ['linear', 'rf', 'svr'],
            'metrics': ['r2', 'mae', 'mse', 'rmse'],
            'cv_folds': 5,
            'value_field': FIELD_CONFIG['regression_target']
        }
    },
    columns=FEATURE_COLUMNS,
    normalize=True,
    use_dask=False,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Classification: All models + 5-fold CV")
print(f"  Regression: All models + 5-fold CV")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_ensemble',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Store metrics
if result_s3.metrics:
    metrics_s3 = result_s3.metrics
    if 'classification' in metrics_s3:
        print(f"\n📊 Classification Results (CV-averaged):")
        for model in ['logistic', 'rf', 'svm']:
            if model in metrics_s3['classification']:
                print(f"\n   {model.upper()}:")
                for metric, value in metrics_s3['classification'][model].items():
                    if isinstance(value, (int, float)):
                        print(f"     {metric}: {value:.4f}")
    
    if 'regression' in metrics_s3:
        print(f"\n📊 Regression Results (CV-averaged):")
        for model in ['linear', 'rf', 'svr']:
            if model in metrics_s3['regression']:
                print(f"\n   {model.upper()}:")
                for metric, value in metrics_s3['regression'][model].items():
                    if isinstance(value, (int, float)):
                        print(f"     {metric}: {value:.4f}")

## Step 4: Strategy Comparison

**How to use:**
- Review execution times across all 3 strategies
- Compare model performance (accuracy, R²)
- Identify optimal strategy for your use case

**What you'll see:**
- Execution time comparison (seconds)
- Total processing time
- Model count per strategy
- Performance ranking

**Strategy selection guide:**
- **Basic**: Fast baseline (2-5s) - good for quick checks
- **Advanced**: Robust models (5-15s) - good for production
- **Ensemble**: Most reliable (10-30s) - good for reporting

**Note:** Higher execution time generally yields more reliable metrics

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Basic):     {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Advanced):  {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Ensemble):  {elapsed_s3:6.2f}s")
print(f"  Total:                  {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Model Coverage:")
print(f"  Strategy 1: 2 models (logistic, linear)")
print(f"  Strategy 2: 4 models (rf, svm, rf, svr)")
print(f"  Strategy 3: 6 models (all models with CV)")

print(f"\n🎯 Performance Summary:")
if 'metrics_s1' in locals() and 'metrics_s2' in locals() and 'metrics_s3' in locals():
    # Classification comparison
    if 'classification' in metrics_s1 and 'classification' in metrics_s3:
        print(f"\n  Classification (Accuracy):")
        if 'logistic' in metrics_s1['classification']:
            acc_s1 = metrics_s1['classification']['logistic'].get('accuracy', 0)
            print(f"    Strategy 1 (Logistic): {acc_s1:.4f}")
        if 'rf' in metrics_s2['classification']:
            acc_s2 = metrics_s2['classification']['rf'].get('accuracy', 0)
            print(f"    Strategy 2 (RF):       {acc_s2:.4f}")
        if 'rf' in metrics_s3['classification']:
            acc_s3 = metrics_s3['classification']['rf'].get('accuracy', 0)
            print(f"    Strategy 3 (RF+CV):    {acc_s3:.4f}")
    
    # Regression comparison
    if 'regression' in metrics_s1 and 'regression' in metrics_s3:
        print(f"\n  Regression (R²):")
        if 'linear' in metrics_s1['regression']:
            r2_s1 = metrics_s1['regression']['linear'].get('r2', 0)
            print(f"    Strategy 1 (Linear):   {r2_s1:.4f}")
        if 'rf' in metrics_s2['regression']:
            r2_s2 = metrics_s2['regression']['rf'].get('r2', 0)
            print(f"    Strategy 2 (RF):       {r2_s2:.4f}")
        if 'rf' in metrics_s3['regression']:
            r2_s3 = metrics_s3['regression']['rf'].get('r2', 0)
            print(f"    Strategy 3 (RF+CV):    {r2_s3:.4f}")

## Step 5: Detailed Artifact Review

**How to use:**
- Review all generated artifacts from each strategy
- Auto-loads NEWEST files from each category
- Displays visualizations inline for easy review

**What you'll see (per strategy):**
1. **Metrics JSON**: Model performance metrics (accuracy, F1, R², etc.)
2. **Visualizations**: Performance charts and PR curves (first 2 displayed)

**Note:** Only newest files shown. Multiple test runs create versions - older artifacts excluded automatically

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_basic', 'Strategy 1: Basic Models'),
    ('strategy2_advanced', 'Strategy 2: Advanced Models'),
    ('strategy3_ensemble', 'Strategy 3: Ensemble Analysis')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display classification metrics
                if 'classification' in metrics:
                    print(f"\n   Classification:")
                    for model, model_metrics in metrics['classification'].items():
                        print(f"     {model.upper()}:")
                        for k, v in model_metrics.items():
                            if isinstance(v, (int, float)):
                                print(f"       {k}: {v:.4f}")
                
                # Display regression metrics
                if 'regression' in metrics:
                    print(f"\n   Regression:")
                    for model, model_metrics in metrics['regression'].items():
                        print(f"     {model.upper()}:")
                        for k, v in model_metrics.items():
                            if isinstance(v, (int, float)):
                                print(f"       {k}: {v:.4f}")
                
                # Performance
                if 'duration_seconds' in metrics:
                    print(f"\n   Duration: {metrics['duration_seconds']:.4f}s")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Privacy-Utility Tradeoff Analysis

**How to use:**
- Analyze relationship between privacy noise and utility preservation
- Compare original vs transformed performance
- Run AFTER all 3 strategies complete

**What you'll see:**
- Utility preservation ratios (closer to 1.0 = better)
- Performance degradation percentages
- Noise impact analysis
- Optimal tradeoff recommendations

**Interpretation:**
- **>95% preservation**: Excellent utility retention
- **85-95% preservation**: Good balance
- **<85% preservation**: High privacy cost

**Note:** Lower degradation indicates better privacy-preserving transformation

In [ ]:
print("\n" + "=" * 80)
print("⚖️  PRIVACY-UTILITY TRADEOFF ANALYSIS")
print("=" * 80 + "\n")

# Calculate noise impact
print("📊 Noise Impact on Features:")
print("-" * 80)
for col in FEATURE_COLUMNS:
    orig_mean = original_df[col].mean()
    trans_mean = transformed_df[col].mean()
    diff_pct = abs(trans_mean - orig_mean) / orig_mean * 100
    print(f"  {col:20s}: {diff_pct:6.2f}% change")

# Utility preservation analysis
print(f"\n📈 Utility Preservation (Best Models):")
print("-" * 80)

if 'metrics_s3' in locals():
    # Classification
    if 'classification' in metrics_s3:
        best_acc = 0
        for model, model_metrics in metrics_s3['classification'].items():
            if 'accuracy' in model_metrics:
                if model_metrics['accuracy'] > best_acc:
                    best_acc = model_metrics['accuracy']
        
        # Assume baseline accuracy of 0.9 for original (you can adjust)
        baseline_acc = 0.90
        preservation = (best_acc / baseline_acc) * 100
        degradation = 100 - preservation
        
        print(f"\n  Classification (Accuracy):")
        print(f"    Baseline (est.):      {baseline_acc:.4f}")
        print(f"    Best transformed:     {best_acc:.4f}")
        print(f"    Preservation:         {preservation:.2f}%")
        print(f"    Degradation:          {degradation:.2f}%")
    
    # Regression
    if 'regression' in metrics_s3:
        best_r2 = 0
        for model, model_metrics in metrics_s3['regression'].items():
            if 'r2' in model_metrics:
                if model_metrics['r2'] > best_r2:
                    best_r2 = model_metrics['r2']
        
        # Assume baseline R² of 0.85 for original
        baseline_r2 = 0.85
        preservation = (best_r2 / baseline_r2) * 100
        degradation = 100 - preservation
        
        print(f"\n  Regression (R²):")
        print(f"    Baseline (est.):      {baseline_r2:.4f}")
        print(f"    Best transformed:     {best_r2:.4f}")
        print(f"    Preservation:         {preservation:.2f}%")
        print(f"    Degradation:          {degradation:.2f}%")

print(f"\n💡 Interpretation:")
print(f"  >95% preservation:  Excellent utility retention")
print(f"  85-95% preservation: Good balance")
print(f"  <85% preservation:  High privacy cost")

## Step 7: Export Final Results

**How to use:**
- Export comprehensive results summary
- Saves metrics comparison table
- Run AFTER all strategies complete

**What you'll see:**
- Export confirmation (file path)
- Summary statistics per strategy
- Best performing models per metric type

**Output structure:**
```
advanced_output/
├── strategy1_basic/
├── strategy2_advanced/
├── strategy3_ensemble/
└── utility_metrics_summary.json
```

In [ ]:
print("=" * 80)
print("📦 EXPORTING FINAL RESULTS")
print("=" * 80)

# Create summary
summary = {
    "timestamp": datetime.now().isoformat(),
    "dataset": {
        "records": len(original_df),
        "features": FEATURE_COLUMNS,
        "targets": list(FIELD_CONFIG.values())
    },
    "strategies": {
        "strategy1_basic": {
            "execution_time": elapsed_s1,
            "models": 2,
            "cv_folds": 1
        },
        "strategy2_advanced": {
            "execution_time": elapsed_s2,
            "models": 4,
            "cv_folds": 1
        },
        "strategy3_ensemble": {
            "execution_time": elapsed_s3,
            "models": 6,
            "cv_folds": 5
        }
    }
}

# Add metrics if available
if 'metrics_s1' in locals():
    summary['strategies']['strategy1_basic']['metrics'] = metrics_s1
if 'metrics_s2' in locals():
    summary['strategies']['strategy2_advanced']['metrics'] = metrics_s2
if 'metrics_s3' in locals():
    summary['strategies']['strategy3_ensemble']['metrics'] = metrics_s3

# Save summary
summary_path = task_dir / 'utility_metrics_summary.json'
try:
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\n✅ Saved summary: {summary_path}")
except PermissionError:
    print(f"\n⚠️  Cannot save (file may be open): {summary_path}")

print(f"\n📂 All files saved to: {task_dir}")
print(f"⏱️  Total processing time: {elapsed_s1 + elapsed_s2 + elapsed_s3:.2f}s")

print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Multiple models evaluated (logistic, linear, RF, SVM/SVR)
- ✅ Privacy-utility tradeoff analyzed
- ✅ Comprehensive metrics calculated
- ✅ Production-ready evaluation pipeline

**Key takeaways:**
- Basic models provide quick baseline (2-5s)
- Advanced models offer robust evaluation (5-15s)
- Ensemble with CV gives most reliable results (10-30s)
- Privacy noise impact typically 5-10% on features
- Utility preservation >85% indicates good transformation

**Strategy recommendations:**
- **Use Strategy 1** when: Quick checks, development iteration, limited compute
- **Use Strategy 2** when: Production deployment, balanced speed/accuracy
- **Use Strategy 3** when: Final evaluation, reporting, research publications

**Next steps:**
- Test with your own datasets
- Tune privacy parameters to optimize tradeoff
- Integrate with privacy profiling results
- Deploy best-performing strategy to production

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)